In [2]:
import pandas as pd
import numpy as np
from itertools import permutations
from statsmodels.tsa.stattools import coint

In [ ]:
filepath = "/Users/benjaminsumacbookm4/Desktop/Pairs Trading/data/IntradayMarketDataKO_PEP"
data = pd.read_parquet(filepath)

In [4]:
data.head()

,symbol,timestamp,close
0,KO,2020-11-23 09:00:00+00:00,3.968025
1,KO,2020-11-23 09:23:00+00:00,3.971989
2,KO,2020-11-23 10:01:00+00:00,3.970292
3,KO,2020-11-23 10:30:00+00:00,3.972930
4,KO,2020-11-23 11:11:00+00:00,3.972177


In [5]:
data.tail()

,symbol,timestamp,close
866142,PEP,2024-11-20 21:31:00+00:00,5.068464
866143,PEP,2024-11-20 21:46:00+00:00,5.068527
866144,PEP,2024-11-20 21:54:00+00:00,5.068464
866145,PEP,2024-11-20 22:08:00+00:00,5.067960
866146,PEP,2024-11-20 22:12:00+00:00,5.067960


In [6]:
# Show timestamps in New York time so the session is 09:30–16:00 year-round (handles DST)
data["timestamp"] = data["timestamp"].dt.tz_convert("America/New_York")

prices = (
    data.pivot(index="timestamp", columns="symbol", values="close")
        .between_time("09:30", "15:59") # Normalizes time to regular business hours
        .dropna()
)
prices.head()
prices["KO"].shape

(389856,)

- score = more negative means more likely to be not cointegrated
- pvalue = Smaller value rejects null hypothesis of lack of cointegration
- crit_value = depends if 0, 1, or 2. Rejects lack of cointegration at 99% probability if score < crit_value[0], 95% if score < crit_value[1], 90% if score < crit_value[2]

In [8]:
score, pvalue, crit_value = coint( 
    prices["KO"], 
    prices["PEP"],
    trend="c",
    autolag="AIC"
)

print(f"Score: {score}")
print(f"P-value: {pvalue}")
print(f"Critical Value {crit_value}")

Score: -1.6864194663749539
P-value: 0.6830785913387312
Critical Value [-3.89646809 -3.33614567 -3.04446088]


In [1]:
if score < crit_value[0]:
    print("There is a 99% chance of cointegration!")
elif score < crit_value[1]:
    print("There is a 95% chance of cointegration!")
elif score < crit_value[2]:
    print("There is a 90% chance of cointegration!")
else:
    print("There is no statistically significant cointegration in the dataset. ")

NameError: name 'score' is not defined